In [11]:
import mach
import time

In [16]:
# Row Row Row Your Boat — 6/8 time, scheduled with mach time units
params = mach.EngineInitParams()
params.sample_rate = 48000
params.block_size = 512
params.max_node_pool_size = 100
params.bpm = 120.0

engine = mach.Engine(params)
osc = engine.add_node("wavetable_oscillator")
osc["amplitude"] = 0.0
osc["waveform"] = mach.Waveform.SINE

out = engine.get_master_output()
engine.connect(osc, out)

# frequencies
C4, D4, E4, F4, G4, C5 = 261.63, 293.66, 329.63, 349.23, 392.00, 523.25

# 6/8 time — eighth note = 0.5 beats, dotted quarter = 1.5, dotted half = 3.0
# (beat_offset, frequency, duration_in_beats)
melody = [
    # "Row row row your boat"
    (0,    C4, 1.5),  (1.5,  C4, 1.5),  (3,    C4, 1.0),  (4,    D4, 0.5),  (4.5,  E4, 1.5),
    # "Gently down the stream"
    (6,    E4, 1.0),  (7,    D4, 0.5),   (7.5,  E4, 1.0),  (8.5,  F4, 0.5),  (9,    G4, 3.0),
    # "Merrily merrily merrily merrily"
    (12,   C5, 0.5),  (12.5, C5, 0.5),  (13,   C5, 0.5),
    (13.5, G4, 0.5),  (14,   G4, 0.5),  (14.5, G4, 0.5),
    (15,   E4, 0.5),  (15.5, E4, 0.5),  (16,   E4, 0.5),
    (16.5, C4, 0.5),  (17,   C4, 0.5),  (17.5, C4, 0.5),
    # "Life is but a dream"
    (18,   G4, 1.0),  (19,   F4, 0.5),  (19.5, E4, 1.0),  (20.5, D4, 0.5),  (21,   C4, 3.0),
]

GATE = 0.9  # proportion of note duration to sound (rest = articulation gap)

for beat, freq, dur in melody:
    engine.schedule_set_param(osc, "frequency", freq, mach.Beats(beat))
    engine.schedule_set_param(osc, "amplitude", 0.3, mach.Beats(beat))
    engine.schedule_set_param(osc, "amplitude", 0.0, mach.Beats(beat + dur * GATE))

engine.play()
engine.sleep(mach.Beats(24.0))
engine.stop()